# 04 - Feature Engineering

## Objective

The objective of this notebook is to create and transform meaningful features that can improve the ability of machine learning models to predict student dropout risk.

The feature engineering techniques are selected based on the patterns and relationships identified during exploratory data analysis.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Load Cleaned Dataset

In [2]:
df = pd.read_csv("../data/processed/student_dropout_cleaned.csv")

df.head()

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,24.4,Male,29740.5,Yes,4.00,82.2,2,38.6,No,No,5.5,1.78,1.77,1.77,Year 1,CS,High School,1
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [3]:
df.shape

(10000, 18)

## Technique 1 — Academic Performance Score

The `GPA`, `Semester_GPA`, and `CGPA` features represent different aspects of academic performance and were found to be strongly correlated during EDA.

An aggregate `Academic_Performance_Score` is created to represent overall academic performance using these related academic measures.

In [4]:
df["Academic_Performance_Score"] = (
    df["GPA"] +
    df["Semester_GPA"] +
    df["CGPA"]
) / 3

In [5]:
df[
    ["GPA", "Semester_GPA", "CGPA", "Academic_Performance_Score"]
].head()

,GPA,Semester_GPA,CGPA,Academic_Performance_Score
0,0.96,0.90,0.90,0.920000
1,1.28,1.20,1.19,1.223333
2,1.68,1.32,1.32,1.440000
3,1.78,1.77,1.77,1.773333
4,1.48,0.91,0.87,1.086667


## Technique 2 — Study Attendance Interaction

A combined `Study_Attendance_Score` is created by considering both daily study hours and attendance rate.

This interaction captures the student's overall study engagement more effectively than either feature individually.

In [6]:
df["Study_Attendance_Score"] = (
    df["Study_Hours_per_Day"] * df["Attendance_Rate"] / 100
)

## Technique 3 — Stress Level Binning

The continuous `Stress_Index` feature is converted into three meaningful categories: Low, Medium, and High.

Binning can help the model capture different levels of stress-related risk rather than relying only on the raw numerical value.

In [7]:
df["Stress_Level"] = pd.cut(
    df["Stress_Index"],
    bins=[0, 3, 7, 10],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

In [8]:
df["Stress_Level"].value_counts()

Stress_Level
Medium    7362
High      1839
Low        799
Name: count, dtype: int64

## Technique 4 — Log Transformation of Family Income

`Family_Income` showed a strongly right-skewed distribution during EDA.

A logarithmic transformation is applied to reduce the effect of extreme high-income values and produce a more balanced distribution.

In [9]:
df["Log_Family_Income"] = np.log1p(df["Family_Income"])

In [10]:
df[
    ["Family_Income", "Log_Family_Income"]
].head()

,Family_Income,Log_Family_Income
0,25000.0,10.126671
1,25000.0,10.126671
2,40183.0,10.601224
3,29740.5,10.300299
4,25319.0,10.139350


## Technique 5 — Travel-to-Study Ratio

A `Travel_Study_Ratio` feature is created by comparing daily travel time with daily study time.

This feature provides an additional measure of how much time spent travelling may compare with the student's available study time.

In [11]:
df["Travel_Study_Ratio"] = (
    df["Travel_Time_Minutes"] /
    (df["Study_Hours_per_Day"] * 60)
)

In [12]:
df[
    [
        "Travel_Time_Minutes",
        "Study_Hours_per_Day",
        "Travel_Study_Ratio"
    ]
].head()

,Travel_Time_Minutes,Study_Hours_per_Day,Travel_Study_Ratio
0,20.4,3.36,0.101190
1,44.0,4.30,0.170543
2,48.9,4.40,0.185227
3,38.6,4.00,0.160833
4,23.0,4.19,0.091488


## Technique 6 — Assignment Delay Binning

`Assignment_Delay_Days` is transformed into three categories: Low, Moderate, and High delay.

This transformation represents assignment completion behavior in a more interpretable form and may help identify students with higher academic risk.

In [13]:
df["Assignment_Delay_Level"] = pd.cut(
    df["Assignment_Delay_Days"],
    bins=[-1, 1, 3, 8],
    labels=["Low", "Moderate", "High"]
)

In [14]:
df["Assignment_Delay_Level"].value_counts()

Assignment_Delay_Level
Low         4631
Moderate    4248
High        1121
Name: count, dtype: int64

## Check New Features

In [15]:
new_features = [
    "Academic_Performance_Score",
    "Study_Attendance_Score",
    "Stress_Level",
    "Log_Family_Income",
    "Travel_Study_Ratio",
    "Assignment_Delay_Level"
]

df[new_features].head()

,Academic_Performance_Score,Study_Attendance_Score,Stress_Level,Log_Family_Income,Travel_Study_Ratio,Assignment_Delay_Level
0,0.920000,2.89296,Medium,10.126671,0.101190,Moderate
1,1.223333,2.92400,Medium,10.126671,0.170543,Moderate
2,1.440000,3.11960,Medium,10.601224,0.185227,Low
3,1.773333,3.28800,Medium,10.300299,0.160833,Moderate
4,1.086667,3.17183,Medium,10.139350,0.091488,Low


## Check Correlations Again

In [16]:
numerical_features = df.select_dtypes(
    include=["int64", "float64"]
).columns

correlation = df[numerical_features].corr()["Dropout"].sort_values(
    ascending=False
)

correlation

Dropout                       1.000000
Stress_Index                  0.249361
Assignment_Delay_Days         0.082327
Travel_Study_Ratio            0.066470
Travel_Time_Minutes           0.028080
Age                           0.007585
Family_Income                -0.010324
Log_Family_Income            -0.012936
Study_Hours_per_Day          -0.087163
Study_Attendance_Score       -0.131430
Attendance_Rate              -0.163539
CGPA                         -0.444807
Semester_GPA                 -0.445396
Academic_Performance_Score   -0.453835
GPA                          -0.460352
Name: Dropout, dtype: float64

## Final Feature List

In [17]:
df.columns.tolist()

['Age',
 'Gender',
 'Family_Income',
 'Internet_Access',
 'Study_Hours_per_Day',
 'Attendance_Rate',
 'Assignment_Delay_Days',
 'Travel_Time_Minutes',
 'Part_Time_Job',
 'Scholarship',
 'Stress_Index',
 'GPA',
 'Semester_GPA',
 'CGPA',
 'Semester',
 'Department',
 'Parental_Education',
 'Dropout',
 'Academic_Performance_Score',
 'Study_Attendance_Score',
 'Stress_Level',
 'Log_Family_Income',
 'Travel_Study_Ratio',
 'Assignment_Delay_Level']

In [ ]:
# Infinite/NaN check
print(
    "Travel_Study_Ratio — Inf:",
    np.isinf(df["Travel_Study_Ratio"]).sum(),
    "| NaN:",
    df["Travel_Study_Ratio"].isna().sum()
)

# Check the engineered dataset
print("Engineered dataset shape:", df.shape)

# Save the engineered dataset
df.to_csv(
    "../data/processed/student_dropout_engineered.csv",
    index=False
)

print("Engineered dataset saved successfully.")

Travel_Study_Ratio — Inf: 0 | NaN: 0
Engineered dataset shape: (10000, 24)
Engineered dataset saved successfully.


## Feature Engineering Summary

The following meaningful feature engineering techniques were applied:

1. Created an `Academic_Performance_Score` from related academic performance features.
2. Created a `Study_Attendance_Score` to capture combined student engagement.
3. Converted `Stress_Index` into meaningful stress-level categories.
4. Applied logarithmic transformation to `Family_Income` to reduce skewness.
5. Created a `Travel_Study_Ratio` to represent the relationship between travel and study time.
6. Converted `Assignment_Delay_Days` into meaningful delay categories.

These engineered features will be evaluated during model development to determine whether they improve prediction performance.